# PDD Mobility Scanner — Cross-Grade with Trail Segmentation

Pipeline:
1. **SAM 2** segments the trail surface (excludes sky, trees, etc.)
2. **Depth Anything V2** estimates depth
3. Cross-grade is computed from depth values **only on trail pixels**

**Camera:** JX-F37 module (110° HFOV, 60° VFOV, f-theta lens, 1920×1080)

**Requirements:** GPU runtime (Runtime → Change runtime type → T4 GPU)

## Step 1: Install dependencies

In [ ]:
!pip install -q transformers torch pillow matplotlib
!pip install -q SAM-2

## Step 2: Camera parameters

In [ ]:
import numpy as np
import torch

# JX-F37 camera module specs
IMG_W, IMG_H = 1920, 1080
HFOV_DEG = 110.0
VFOV_DEG = 60.0
CX, CY = IMG_W / 2, IMG_H / 2

# f-theta focal length: r = f * θ
# At horizontal edge: 960 px = f * radians(55°)  →  f ≈ 1000 px
F_THETA = (IMG_W / 2) / np.radians(HFOV_DEG / 2)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"f-theta focal length: {F_THETA:.1f} px")
print(f"Device: {DEVICE}")

## Step 3: Upload hike photos

In [ ]:
from google.colab import files
from PIL import Image

uploaded = files.upload()
image_files = sorted(uploaded.keys())

images = {}
for fname in image_files:
    images[fname] = Image.open(fname).convert("RGB")

print(f"Uploaded {len(image_files)} image(s): {image_files}")

## Step 4: Segment trail with SAM 2

We prompt SAM 2 with a point at the **bottom-center** of each image
(where the trail surface is). SAM 2 returns the connected region
around that point — the trail mask.

In [ ]:
from sam2.sam2_image_predictor import SAM2ImagePredictor

sam_predictor = SAM2ImagePredictor.from_pretrained("facebook/sam2.1-hiera-small", device=DEVICE)
print("SAM 2 loaded.")

In [ ]:
trail_masks = {}

# Prompt point: bottom-center of image (where trail is)
prompt_point = np.array([[IMG_W // 2, int(IMG_H * 0.85)]])
prompt_label = np.array([1])  # 1 = foreground

for fname in image_files:
    image_np = np.array(images[fname])

    with torch.inference_mode():
        sam_predictor.set_image(image_np)
        masks, scores, _ = sam_predictor.predict(
            point_coords=prompt_point,
            point_labels=prompt_label,
            multimask_output=True,
        )

    # SAM 2 returns 3 masks at different granularities; pick the best one
    best_idx = scores.argmax()
    trail_masks[fname] = masks[best_idx]  # boolean array (H, W)

    pct = trail_masks[fname].sum() / trail_masks[fname].size * 100
    print(f"{fname}: mask covers {pct:.1f}% of image (score {scores[best_idx]:.3f})")

print(f"\nSegmented {len(trail_masks)} image(s).")

### Visualize trail masks

In [ ]:
import matplotlib.pyplot as plt

for fname in image_files:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].imshow(images[fname])
    # Show prompt point
    axes[0].plot(IMG_W // 2, int(IMG_H * 0.85), "r*", markersize=15)
    axes[0].set_title(f"{fname} (red star = prompt point)")
    axes[0].axis("off")

    # Overlay mask on image
    overlay = np.array(images[fname]).copy()
    mask = trail_masks[fname]
    overlay[~mask] = (overlay[~mask] * 0.3).astype(np.uint8)  # dim non-trail
    overlay[mask, 1] = np.minimum(overlay[mask, 1].astype(int) + 60, 255).astype(np.uint8)  # tint trail green
    axes[1].imshow(overlay)
    axes[1].set_title("Trail mask (green = trail)")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

## Step 5: Run depth estimation

Depth Anything needs the **full image** for context (it uses global scene
understanding). We run it on the complete photo, then apply the trail mask
to the output.

In [ ]:
from transformers import pipeline as hf_pipeline

depth_pipe = hf_pipeline(task="depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf")

results = []
for fname in image_files:
    depth_output = depth_pipe(images[fname])
    depth_array = np.array(depth_output["depth"]).astype(float)

    # Apply trail mask: set non-trail pixels to NaN
    masked_depth = depth_array.copy()
    masked_depth[~trail_masks[fname]] = np.nan

    results.append({
        "filename": fname,
        "image": images[fname],
        "depth_array": depth_array,
        "depth_map": depth_output["depth"],
        "masked_depth": masked_depth,
        "trail_mask": trail_masks[fname],
    })
    valid = np.count_nonzero(~np.isnan(masked_depth))
    print(f"{fname}: {valid} trail pixels with depth")

print(f"\nProcessed {len(results)} image(s).")

## Step 6: Estimate cross-grade (trail pixels only)

Same f-theta projection as before, but now the horizontal band only
averages depth over columns where the trail mask is active.
Columns with no trail pixels are excluded from the line fit.

In [ ]:
def estimate_cross_grade(depth_array, trail_mask=None,
                         band_center_frac=0.80, band_height_frac=0.05):
    """
    Estimate cross-grade from a depth map, optionally masked to trail pixels.
    """
    h, w = depth_array.shape

    # --- 1. Extract horizontal band ---
    band_center = int(h * band_center_frac)
    band_half = max(1, int(h * band_height_frac / 2))
    band_top = max(0, band_center - band_half)
    band_bot = min(h, band_center + band_half)

    band = depth_array[band_top:band_bot, :].copy()

    # Apply trail mask to the band
    if trail_mask is not None:
        mask_band = trail_mask[band_top:band_bot, :]
        band[~mask_band] = np.nan

    # Average depth per column (ignoring NaN)
    with np.errstate(all="ignore"):
        depth_profile = np.nanmean(band, axis=0)

    # Columns that have trail pixels
    valid = ~np.isnan(depth_profile)

    # --- 2. f-theta projection to 3D ---
    x = np.arange(w)
    dx = x - CX
    dy = band_center - CY

    r = np.sqrt(dx**2 + dy**2)
    theta = r / F_THETA

    safe_r = np.where(r > 0, r, 1)
    ray_x = np.sin(theta) * dx / safe_r
    ray_y = np.sin(theta) * dy / safe_r

    X = depth_profile * ray_x
    Y = depth_profile * ray_y

    # --- 3. Fit line through valid trail columns ---
    fit_mask = valid
    if fit_mask.sum() < 10:
        return None  # not enough trail pixels in this band

    coeffs = np.polyfit(X[fit_mask], Y[fit_mask], 1)
    slope_ratio = coeffs[0]
    cross_grade_deg = np.degrees(np.arctan(slope_ratio))
    cross_grade_pct = slope_ratio * 100

    return {
        "cross_grade_deg": cross_grade_deg,
        "cross_grade_pct": cross_grade_pct,
        "depth_profile": depth_profile,
        "X": X,
        "Y": Y,
        "band_top": band_top,
        "band_bot": band_bot,
        "fit_coeffs": coeffs,
        "fit_mask": fit_mask,
    }


for r in results:
    cg = estimate_cross_grade(r["depth_array"], trail_mask=r["trail_mask"])
    r["cross_grade"] = cg
    if cg is None:
        print(f"{r['filename']:30s}  not enough trail pixels in band")
    else:
        sign = "left-low" if cg["cross_grade_deg"] > 0 else "right-low"
        print(f"{r['filename']:30s}  cross-grade: {cg['cross_grade_deg']:+.2f}°  ({cg['cross_grade_pct']:+.1f}%)  [{sign}]")

## Step 7: Visualize

Four panels: original + prompt point, trail mask, masked depth map, cross-section with fit.

In [ ]:
for r in results:
    cg = r["cross_grade"]
    if cg is None:
        continue

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))

    # --- Original photo ---
    axes[0].imshow(r["image"])
    axes[0].plot(IMG_W // 2, int(IMG_H * 0.85), "r*", markersize=12)
    axes[0].set_title(r["filename"])
    axes[0].axis("off")

    # --- Trail mask overlay ---
    overlay = np.array(r["image"]).copy()
    overlay[~r["trail_mask"]] = (overlay[~r["trail_mask"]] * 0.3).astype(np.uint8)
    axes[1].imshow(overlay)
    axes[1].axhline(cg["band_top"], color="cyan", linewidth=1, linestyle="--")
    axes[1].axhline(cg["band_bot"], color="cyan", linewidth=1, linestyle="--")
    axes[1].set_title("Trail mask + band")
    axes[1].axis("off")

    # --- Masked depth map ---
    im = axes[2].imshow(r["masked_depth"], cmap="inferno")
    axes[2].axhline(cg["band_top"], color="cyan", linewidth=1, linestyle="--")
    axes[2].axhline(cg["band_bot"], color="cyan", linewidth=1, linestyle="--")
    axes[2].set_title("Depth (trail only)")
    axes[2].axis("off")
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    # --- Cross-section fit ---
    X, Y, fm = cg["X"], cg["Y"], cg["fit_mask"]
    axes[3].scatter(X[fm], Y[fm], s=1, alpha=0.5, label="trail cross-section")
    x_fit = np.array([X[fm].min(), X[fm].max()])
    y_fit = np.polyval(cg["fit_coeffs"], x_fit)
    axes[3].plot(x_fit, y_fit, "r-", linewidth=2,
                 label=f"cross-grade: {cg['cross_grade_deg']:+.2f}° ({cg['cross_grade_pct']:+.1f}%)")
    axes[3].set_xlabel("X (across trail)")
    axes[3].set_ylabel("Y (vertical)")
    axes[3].set_title("Cross-section")
    axes[3].legend(fontsize=8)
    axes[3].set_aspect("equal")
    axes[3].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## Step 8: Summary statistics

In [ ]:
import pandas as pd

stats = []
for r in results:
    cg = r["cross_grade"]
    trail_pct = r["trail_mask"].sum() / r["trail_mask"].size * 100
    row = {
        "filename": r["filename"],
        "trail_coverage_pct": round(trail_pct, 1),
        "cross_grade_deg": round(cg["cross_grade_deg"], 2) if cg else None,
        "cross_grade_pct": round(cg["cross_grade_pct"], 1) if cg else None,
    }
    stats.append(row)

df_stats = pd.DataFrame(stats)
df_stats

## Step 9: Export

In [ ]:
for r in results:
    # Save depth map
    out_name = r["filename"].rsplit(".", 1)[0] + "_depth.png"
    r["depth_map"].save(out_name)
    print(f"Saved {out_name}")

    # Save trail mask
    mask_name = r["filename"].rsplit(".", 1)[0] + "_trail_mask.png"
    Image.fromarray((r["trail_mask"] * 255).astype(np.uint8)).save(mask_name)
    print(f"Saved {mask_name}")

# Save stats
df_stats.to_csv("cross_grade_stats.csv", index=False)
print("Saved cross_grade_stats.csv")

# Download all
files.download("cross_grade_stats.csv")
for r in results:
    for suffix in ["_depth.png", "_trail_mask.png"]:
        files.download(r["filename"].rsplit(".", 1)[0] + suffix)